<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/fabiobento/lab-cont-2026-2/blob/main/modulo1_introducao/labs/lab03_estabilidade_e_erro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

# Lab 03 — Estabilidade, Realimentação e Erro em Regime

**Módulo 01 · Semana 4 · Laboratório de Controle Automático (Ifes Guarapari)**

Neste laboratório você vai:
1. Verificar estabilidade BIBO pela posição dos polos (e conferir com Routh no papel);
2. Estabilizar uma planta instável por realimentação;
3. Medir a faixa de estabilidade de um ganho k varrendo os polos de T(s);
4. Medir erros em regime e confrontar com kp, kv, ka.

**Teoria de apoio:** `teoria_modulo1.md`, §1.3 (BIBO, Routh, realimentação, erro em regime).

In [ ]:
# %% IMPORTS (rode esta célula primeiro)
!pip install --quiet control==0.10.2
import control as ct
import numpy as np
import matplotlib.pyplot as plt
s = ct.tf('s')
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.grid'] = True
print('python-control', ct.__version__)

## Parte 1 — BIBO: os polos mandam

Reproduza os dois exemplos divergentes da teoria (§1.3.1):
G(s) = 1/s com degrau (saída = rampa) e G(s) = 1/(s²+1) com entrada cos(t) (ressonância).

In [ ]:
t = np.linspace(0, 20, 2000)

# 1/s com degrau -> rampa diverge
tt, y = ct.step_response(1/s, t)
plt.plot(tt, y); plt.title('G=1/s + degrau ⇒ y = t (diverge)'); plt.show()

# 1/(s^2+1) com cos(t) -> ressonância
G = 1/(s**2 + 1)
tt, y = ct.forced_response(G, t, np.cos(t))
plt.plot(tt, y); plt.title('G=1/(s²+1) + cos(t) ⇒ (t/2)·sen(t) (diverge)'); plt.show()
print('polos:', ct.poles(1/s), ct.poles(G))

## Parte 2 — Realimentação estabilizando

G(s) = 1/(s−1) é instável. Com controle P (u = k·e) e realimentação unitária:
T(s) = k/(s − 1 + k) → estável ⇔ k > 1 (§1.3.3). Varra k = 0.5, 1, 2, 10 e observe.

In [ ]:
t = np.linspace(0, 8, 800)
G = 1/(s - 1)
for k in [0.5, 1, 2, 10]:
    T = ct.feedback(k*G)   # realimentação unitária negativa
    print(f'k={k:>4}: polos de T(s) = {np.round(ct.poles(T),3)}')
    if k > 1:
        tt, y = ct.step_response(T, t)
        plt.plot(tt, y, label=f'k={k}')
tt, y = ct.step_response(G, t)  # malha aberta, para comparação
plt.plot(tt, np.clip(y, -5, 30), 'k--', label='MA (instável)')
plt.legend(); plt.title('Realimentação estabilizando G=1/(s−1)')
plt.xlabel('t (s)'); plt.ylabel('y(t)'); plt.show()

## Parte 3 — Faixa de estabilidade (varredura numérica × Routh)

No papel você provou: T(s) com denominador s³ + 6s² + 11s + (6 + k) é estável
⇔ **−6 < k < 60** (§1.3.4). Confira numericamente varrendo k e vendo quando os
polos cruzam o eixo imaginário.

In [ ]:
def polos_T(k):
    return np.roots([1, 6, 11, 6 + k])

for k in [-6, -5.9, 0, 10, 59.9, 60, 60.1]:
    p = polos_T(k)
    estavel = all(np.real(p) < 0)
    print(f'k={k:>6}: Re máx = {max(np.real(p)):+.5f}  {"estável" if estavel else "INSTÁVEL/marginal"}')

print()
print('polos no limite k=60:', np.round(polos_T(60), 4))   # esperado: -6, ±j√11 ≈ ±j3.3166
print('polos no limite k=-6:', np.round(polos_T(-6), 4))   # esperado: polo na origem

## Parte 4 — Erro em regime: medindo na simulação

Para L(s) = k/[(s+1)(s+2)] (tipo 0) com k = 10, a teoria (§1.3.6) prevê:
kp = 5 ⇒ ess(degrau) = 1/6 ≈ 0,167; kv = 0 ⇒ ess(rampa) = ∞.
Meça os dois na simulação.

In [ ]:
k = 10
L = k/((s+1)*(s+2))
T = ct.feedback(L)
t = np.linspace(0, 30, 3000)

# Degrau
tt, y = ct.step_response(T, t)
tt, y = np.asarray(tt), np.asarray(y)
ess = 1 - y[-1]
print(f'ess(degrau) medido = {ess:.4f}   teórico = {1/(1+k/2):.4f}')

# Rampa (u = t)
tt, y = ct.forced_response(T, t, t)
tt, y = np.asarray(tt), np.asarray(y)
plt.plot(tt, tt, 'k--', label='r(t) = t')
plt.plot(tt, y, label='y(t)')
plt.legend(); plt.title('Tipo 0 seguindo rampa: erro cresce sem limite'); plt.show()
print(f'erro(rampa) em t=30s: {30 - y[-1]:.2f} e crescendo (teórico: infinito)')

# ATENÇÃO — a armadilha do TVF (§1.3.5):
Gtrap = (s**2 + 29*s + 208)/(s**3 + 6*s**2 + 10*s + 208)
print('\nArmadilha: polos de Gtrap =', np.round(ct.poles(Gtrap), 3), '-> instável, TVF NÃO vale!')

## Parte 5 — Sua vez

**Exercício L3.1.** Para G(s) = 10/(s³ + 11s² + 8s − 20) com controle k, determine
a faixa de estabilidade **numericamente** (varredura como na Parte 3) e confira com o
Routh do papel (gabarito: 2 < k < 10,8 — ver Exercício 1.3.6 dos exercícios resolvidos).

**Exercício L3.2.** Sensibilidade: para a planta G = A (ganho puro) com kA = 10,
simule MA e MF com A nominal e com A 20% maior. Meça a variação relativa da saída
em cada caso e confira os valores da teoria (§1.3.3: 16,7% × 1,5%).

**Exercício L3.3.** Acrescente um integrador ao sistema da Parte 4
(L₂(s) = 10·(s+3)/[s(s+1)(s+2)], tipo 1). Preveja ess(degrau) e ess(rampa) pelas
fórmulas de kp e kv, depois meça na simulação. Bateram?